**In this Notebook we will do the following:**

- Construct the Pauli-Transfer-Matrix (PTM) of multiple Circuits
    - Memory Round: Surface Code
    - Lattice Surgery

- Invert the PTM to get an estimate of the Logical Operator

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys, pathlib
repo_root = pathlib.Path.cwd()

# If running from the playground directory, move up one level to the repo root
if repo_root.name == 'playground':
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))
print('Inserted repo root into sys.path:', repo_root)

Inserted repo root into sys.path: c:\Users\f.spreemann\qecsim-work


In [3]:
from src.codes.surface_code_rotated.builder import SurfaceBuilder
from src.core.data_models import NoiseParameters
from src.tools.qem_estimator.logical_level.calc_ptm import PTMCalculator
from src.core.data_models import PTMCircuits

import stim

**PTM Calculation: Surface Code Memory Round**

In [15]:
#########################
# Construct Noise Class #
#########################
noise = 0

noise_class = NoiseParameters(before_round_depol = noise,
    before_m_flip_prob = noise,
    after_r_flip = noise,
    after_c_depol_prob = noise,
    after_c_pauli_channel_prob = noise)

############
# Circuits #
############

circuits : dict[str, tuple[stim.Circuit, list[int]]] = {}

for pauli_input in ["X", "Z"]:
    for pauli_output in ["X", "Z"]:

        if pauli_input == "X":
            pauli_state_init = "+"
        elif pauli_input == "Y":
            pauli_state_init = "+i"
        elif pauli_input == "Z":
            pauli_state_init = "0"

        # Building the circuit for the given input-output Pauli combination
        print(f"Constructing circuit for input {pauli_input} and output {pauli_output}")
        builder = SurfaceBuilder(distance = 3, state_init=pauli_state_init, log_obs = pauli_output, noise = noise_class)
        circuit = builder.build_circuit()
        measurement_records = builder.get_logical_meas_rec()
        print(f"Circuit for input {pauli_input} and output {pauli_output} constructed")

        # Adding the circuit and measurement record to the PTM Circuits data model
        circuits[f"{pauli_input}->{pauli_output}"] = (circuit, measurement_records)

############################
# Calculate the PTM-Matrix #
############################

ptm_calculator = PTMCalculator(PTMCircuits(circuits=circuits), samples=10_000)
ptm_matrix_surface = ptm_calculator.calc_ptm()

ptm_matrix_surface

Constructing circuit for input X and output X
Circuit for input X and output X constructed
Constructing circuit for input X and output Z
Circuit for input X and output Z constructed
Constructing circuit for input Z and output X
Circuit for input Z and output X constructed
Constructing circuit for input Z and output Z
Circuit for input Z and output Z constructed


array([[ 1.00e+00,  0.00e+00, -1.72e-02],
       [ 0.00e+00,  0.00e+00,  0.00e+00],
       [-6.00e-04,  0.00e+00,  1.00e+00]])